# QML Intrusion Detection — NSL-KDD Dataset
> Implementation of: *Kalinin & Krundyshev (2023) — Security intrusion detection using quantum machine learning techniques*
>
> Dataset: **NSL-KDD** (Canadian Institute for Cybersecurity)

---
## Pipeline Overview
1. Load & preprocess NSL-KDD (41 features → one-hot → MinMax scale)
2. Encode classical features → qubit rotation circuits
3. Train **Classical SVM** baseline
4. Train **QSVM** (quantum kernel via Cirq simulator)
5. Evaluate: confusion matrix, ROC curves, accuracy comparison
6. *(Optional)* QCNN training

## 0. Setup & Imports

In [ ]:
# Install dependencies (run once)
# !pip install numpy pandas scikit-learn matplotlib seaborn cirq tqdm

import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

from data.nslkdd_loader import NSLKDDLoader
from encoder.qubit_encoder import QuantumEncoder
from models.qsvm_model import QSVMDetector, SVMBaseline
from models.qcnn_model import QCNNDetector, CNNBaseline
from utils.evaluation import Evaluator

print('All imports OK')

## 1. Load NSL-KDD Dataset

In [ ]:
loader = NSLKDDLoader(data_dir='data/nslkdd')
NSLKDDLoader.describe()

# Auto-download if not present
loader.download()

In [ ]:
# Load full training set
X_train_full, y_train_full = loader.load_train()
X_test_full,  y_test_full  = loader.load_test()

print(f'Train: {X_train_full.shape}  |  Test: {X_test_full.shape}')
print(f'Features: {len(loader.feature_names)}')

In [ ]:
# Class distribution
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, y, title in zip(axes, [y_train_full, y_test_full], ['Train', 'Test']):
    classes, counts = np.unique(y, return_counts=True)
    colors = plt.cm.Set2(np.linspace(0, 1, len(classes)))
    bars = ax.bar(classes, counts, color=colors, edgecolor='white', linewidth=1.5)
    ax.set_title(f'NSL-KDD {title} Set — Class Distribution', fontsize=13, fontweight='bold')
    ax.set_ylabel('Count')
    ax.set_xlabel('Attack Category')
    for bar, count in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
                f'{count:,}', ha='center', va='bottom', fontsize=10)
    ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('results/nslkdd_class_distribution.png', dpi=150)
plt.show()

## 2. Exploratory Data Analysis
### Feature correlation heatmap (top 20 numeric features)

In [ ]:
# Use original (pre-OHE) data for EDA
from data.nslkdd_loader import COL_NAMES, CATEGORICAL_COLS
loader_eda = NSLKDDLoader(data_dir='data/nslkdd')
df_raw = pd.read_csv('data/nslkdd/KDDTrain+.txt', header=None, names=COL_NAMES)

numeric_cols = [c for c in COL_NAMES if c not in CATEGORICAL_COLS + ['label', 'difficulty_level']]
top20 = numeric_cols[:20]

fig, ax = plt.subplots(figsize=(12, 9))
corr = df_raw[top20].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=False, cmap='coolwarm', center=0,
            ax=ax, linewidths=0.3, vmin=-1, vmax=1)
ax.set_title('Feature Correlation — NSL-KDD (top 20 numeric features)', fontsize=13)
plt.tight_layout()
plt.savefig('results/nslkdd_correlation.png', dpi=150)
plt.show()

## 3. Quantum Encoding Demo

In [ ]:
# Take a small sample for encoding demo
# NSL-KDD features are already scaled to [0, π] by the loader
enc = QuantumEncoder(backend='cirq', fit_scaler=False)   # already scaled

sample_5 = X_train_full[:5]
circuits = enc.encode_batch(sample_5, verbose=True)

print('\nFirst circuit (first 8 Rx gates):')
import cirq
q = cirq.GridQubit(1, 1)
demo_circuit = cirq.Circuit()
for val in sample_5[0, :8]:
    demo_circuit.append(cirq.rx(float(val))(q))
print(demo_circuit)

## 4. Model Training

### Subset selection
The QML simulator is exact but slower than real quantum hardware. We use a stratified subset for training (matches the GitHub repo's approach). Scale up `N_SAMPLES` as needed.

In [ ]:
# ── CONFIGURATION ──────────────────────────────────────────────
N_SAMPLES  = 500    # increase for better accuracy (1000, 2000, 5000 ...)
N_QUBITS   = 4     # qubits for QSVM/QCNN (4 = fast, 8 = more expressive)
TEST_SIZE  = 0.2
RANDOM_STATE = 42
# ───────────────────────────────────────────────────────────────

# Stratified sample from train set
X_s, y_s = loader.load_sample(n=N_SAMPLES, random_state=RANDOM_STATE)

X_train, X_val, y_train, y_val = train_test_split(
    X_s, y_s, test_size=TEST_SIZE, stratify=y_s, random_state=RANDOM_STATE)

print(f'Training on {len(X_train)} samples, validating on {len(X_val)}')
print(f'Classes: {sorted(set(y_train))}')

### 4a. Classical SVM Baseline

In [ ]:
svm = SVMBaseline(kernel='rbf', C=1.0)
svm.fit(X_train, y_train)
print('\n=== Classical SVM Report ===')
print(svm.report(X_val, y_val))

### 4b. QSVM (Quantum Kernel — Cirq Simulator)

In [ ]:
qsvm = QSVMDetector(backend='cirq_sim', n_qubits=N_QUBITS, C=1.0)
qsvm.fit(X_train, y_train)
print('\n=== QSVM Report ===')
print(qsvm.report(X_val, y_val))

## 5. Evaluation — Confusion Matrices & ROC Curves

In [ ]:
import os
os.makedirs('results', exist_ok=True)

CLASS_NAMES = ['DoS', 'Normal', 'Probe', 'R2L', 'U2R']
ev = Evaluator(class_names=CLASS_NAMES, save_dir='results')

# Encode integer labels
y_val_enc  = svm.le.transform(y_val)
y_pred_svm = svm.predict(X_val)

# SVM confusion
ev.plot_confusion(y_val_enc, y_pred_svm, title='SVM (NSL-KDD)')

# SVM ROC
try:
    y_prob_svm = svm.model.predict_proba(svm._preprocess(X_val))
    ev.plot_roc(y_val_enc, y_prob_svm, title='SVM (NSL-KDD)')
    ev.plot_combined(y_val_enc, y_pred_svm, y_prob_svm, title='SVM (NSL-KDD)')
except Exception as e:
    print(f'ROC skipped: {e}')

print('SVM plots saved to results/')

In [ ]:
y_val_enc_q = qsvm.le.transform(y_val)
y_pred_qsvm = qsvm.predict(X_val)

# QSVM confusion
ev.plot_confusion(y_val_enc_q, y_pred_qsvm, title='QSVM (NSL-KDD)')

# QSVM ROC
try:
    y_prob_qsvm = qsvm.predict_proba(X_val)
    ev.plot_roc(y_val_enc_q, y_prob_qsvm, title='QSVM (NSL-KDD)')
    ev.plot_combined(y_val_enc_q, y_pred_qsvm, y_prob_qsvm, title='QSVM (NSL-KDD)')
except Exception as e:
    print(f'ROC skipped: {e}')

print('QSVM plots saved to results/')

## 6. Side-by-Side Accuracy Comparison

In [ ]:
from sklearn.metrics import accuracy_score

results = {
    'SVM (Classical)': accuracy_score(y_val_enc,   y_pred_svm),
    'QSVM (Cirq sim)': accuracy_score(y_val_enc_q, y_pred_qsvm),
}

fig, ax = plt.subplots(figsize=(7, 4))
colors = ['#4C72B0', '#DD8452']
bars = ax.bar(results.keys(), [v * 100 for v in results.values()],
              color=colors, edgecolor='white', linewidth=1.5, width=0.4)
for bar, val in zip(bars, results.values()):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.5,
            f'{val*100:.1f}%', ha='center', fontsize=12, fontweight='bold')
ax.set_ylim(0, 110)
ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('Classical SVM vs QSVM — NSL-KDD', fontsize=13, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
ax.axhline(98, color='green', linestyle='--', alpha=0.6, label='Paper target (98%)')
ax.legend()
plt.tight_layout()
plt.savefig('results/accuracy_comparison_nslkdd.png', dpi=150)
plt.show()

for name, acc in results.items():
    print(f'{name:25s} : {acc*100:.2f}%')

## 7. Training Time vs Dataset Size (Table 4 reproduction)

Reproduces Table 4 from the paper: training time grows slower for QSVM than classical SVM as dataset size increases.

In [ ]:
import time, copy

SIZES = [50, 100, 200, 500, 1000]   # add 2000, 5000 for fuller reproduction

timing = {'SVM': [], 'QSVM': []}

for n in SIZES:
    idx = np.random.choice(len(X_s), min(n, len(X_s)), replace=False)
    Xn, yn = X_s[idx], y_s[idx]

    for label, ModelClass, kwargs in [
        ('SVM',  SVMBaseline,  {}),
        ('QSVM', QSVMDetector, {'backend': 'cirq_sim', 'n_qubits': N_QUBITS}),
    ]:
        m = ModelClass(**kwargs)
        t0 = time.time()
        m.fit(Xn, yn)
        elapsed = time.time() - t0
        timing[label].append(elapsed)
        print(f'  {label:6s}  n={n:>5,}  {elapsed:.2f}s')

# Plot
fig, ax = plt.subplots(figsize=(8, 5))
for label, times in timing.items():
    ax.plot(SIZES, times, marker='o', linewidth=2, label=label)
ax.set_xlabel('Training set size (samples)', fontsize=12)
ax.set_ylabel('Training time (seconds)', fontsize=12)
ax.set_title('Training Time: SVM vs QSVM — NSL-KDD (Table 4)', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('results/training_time_nslkdd.png', dpi=150)
plt.show()

## 8. (Optional) QCNN Training

QCNN is slower than QSVM but achieves better accuracy per the paper. Uncomment to run.

In [ ]:
# Uncomment to run QCNN (takes several minutes)

# qcnn = QCNNDetector(n_qubits=4, n_layers=1, backend='cirq_sim',
#                     epochs=10, batch_size=16)
# qcnn.fit(X_train, y_train)
# print(qcnn.report(X_val, y_val))

# y_pred_qcnn = qcnn.predict(X_val)
# y_val_enc_qc = qcnn.le.transform(y_val)
# ev.plot_confusion(y_val_enc_qc, y_pred_qcnn, title='QCNN (NSL-KDD)')

print('QCNN cell is commented out. Uncomment to run.')

## 9. Full Test Set Evaluation

Run the best model (QSVM) against the official NSL-KDD test set.

In [ ]:
print('Evaluating QSVM on full NSL-KDD test set ...')
print('(Re-training on full training set first)')

# Use a larger sample for production evaluation
X_prod, y_prod = loader.load_sample(n=2000)

qsvm_prod = QSVMDetector(backend='cirq_sim', n_qubits=N_QUBITS)
qsvm_prod.fit(X_prod, y_prod)

# Predict on test set
y_test_pred = qsvm_prod.predict(X_test_full[:500])   # first 500 for speed
y_test_enc  = qsvm_prod.le.transform(y_test_full[:500])

ev.plot_combined(y_test_enc, y_test_pred,
                 np.eye(len(CLASS_NAMES))[y_test_pred],  # dummy probs
                 title='QSVM Test Set (NSL-KDD)')

from sklearn.metrics import accuracy_score, classification_report
print(f'Test accuracy: {accuracy_score(y_test_enc, y_test_pred)*100:.2f}%')
print(classification_report(y_test_enc, y_test_pred,
                             target_names=qsvm_prod.le.classes_,
                             zero_division=0))

---
## Summary

| Model | Dataset | Accuracy (paper) | Notes |
|---|---|---|---|
| Classical SVM | NSL-KDD | ~85–92% | Degrades on big data |
| **QSVM** | **NSL-KDD** | **~98%** | Quantum kernel advantage |
| Classical CNN | NSL-KDD | ~92–96% | Requires TensorFlow |
| **QCNN** | **NSL-KDD** | **~98%** | Best per ROC curves |

**Key insight from the paper:** QSVM and QCNN both achieve ~2× faster training than their classical counterparts on big data inputs (>10⁶ samples), while maintaining 98% accuracy.

### To use real IBM Q hardware instead of simulator:
```python
qsvm = QSVMDetector(backend='qiskit', n_qubits=4)  # requires IBM account
```